# Feature Engineering

## Team Selection

Choose the team, Grand Prix, and season that have already been processed. These parameters define the exact comparison window for the analysis.

In [1]:
team = "Aston Martin"
grand_prix = "Japanese Grand Prix"
year = 2025

## Imports and Configuration

Load dependencies, configure the notebook path, and prepare the modules used in the exploration steps.

In [2]:
import sys
import fastf1
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.load_save_data import *
from src.preprocess_data import *
from src.explore_data import *
from src.feature_engineering import *

## Load Processed Data

Read the cached processed laps for both teammates so the notebook can work from already cleaned telemetry.

In [3]:
session = fastf1.get_session(year, grand_prix, "Q")
session.load()

driver_1, driver_2 = get_drivers(session, team)
segment = get_segment(session, driver_1, driver_2)
turns = get_turns(session)

try:
    lap_1, lap_2 = load_cache(year, grand_prix, segment, driver_1, driver_2)
except Exception as e:
    print(f"Processed data for {year} {grand_prix} {driver_1} and {driver_2} not found: {e}")

req         WARNING 	DEFAULT CACHE ENABLED! (598.78 MB) C:\Users\joseg\AppData\Local\Temp\fastf1
core           INFO 	Loading data for Japanese Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '4', '81', '16', '63', '12', '6', '44', '23', '87', '10', '55', '14', '30', '22', '27', '5', '31', '7', '18']



Processed data for 2025 Japanese Grand Prix ALO and STR found in cache.
Driver 1: ALO, Driver 2: STR, Segment: Q1


## Compute Speed Features

In [4]:
try:
    speed_metrics_1 = analyze_speed_metrics(lap_1["Telemetry"], turns)
    speed_metrics_2 = analyze_speed_metrics(lap_2["Telemetry"], turns)
except Exception as e:
    print(f"Speed metrics analysis failed: {e}")

print(f"Speed metrics for {driver_1}:\n\n{speed_metrics_1}\n")
print(f"Speed metrics for {driver_2}:\n\n{speed_metrics_2}")

Speed metrics for ALO:

    Turn  EntrySpeed  MinimumSpeed   ExitSpeed
0      1  316.703833    226.000000  216.639986
1      2  216.639986    175.000000  189.173638
2      3  237.515098    239.352615  254.000000
3      4  249.564264    225.497194  230.219390
4      5  240.460761    228.000000  230.293099
5      6  239.000000    193.341006  192.376692
6      7  247.666981    248.441726  264.163814
7      8  287.578909    255.677180  253.292756
8      9  220.620398    131.076242  157.962637
9     10  259.656799    260.756940  242.368613
10    11  155.684291     71.959831  129.241177
11    12  232.636890    235.726757  251.000000
12    13  275.989707    227.644628  230.295695
13    14  205.816865    169.863700  197.859438
14    15  301.000000    295.144680  295.579633
15    16  113.141150     88.000000  128.861125
16    17  106.697320    112.183443  199.647626
17    18  224.014809    227.351315  258.261961

Speed metrics for STR:

    Turn  EntrySpeed  MinimumSpeed   ExitSpeed
0      1  3

## Compute Braking features

In [5]:
try:
    braking_zones_1 = analyze_braking_zones(lap_1["Telemetry"], turns)
    braking_zones_2 = analyze_braking_zones(lap_2["Telemetry"], turns)
except Exception as e:
    print(f"Braking zones analysis failed: {e}")
    
print(f"Braking zones for {driver_1}:\n\n: {braking_zones_1}\n")
print(f"Braking zones for {driver_2}:\n\n: {braking_zones_2}")

Braking zones for ALO:

:    Turn  StartDistance  EndDistance  BrakingDuration  BrakingDistance  \
0     1     679.970542   782.977661         1.512405       103.007119   
1     6    1469.691790  1512.611423         0.713592        42.919633   
2     8    2233.661258  2285.164818         0.719429        51.503560   
3     9    2345.252304  2413.923717         1.426009        68.671413   
4    11    2791.616487  2877.455754         2.424912        85.839266   
5    13    3718.680561  3787.351974         0.976355        68.671413   
6    14    3856.023387  3924.694800         1.241569        68.671413   
7    16    5255.203424  5341.042690         2.124844        85.839266   

   BrakingPoint  
0     38.894948  
1     51.817657  
2     32.181384  
3     67.416182  
4     98.052495  
5     66.154317  
6     91.309722  
7    124.411490  

Braking zones for STR:

:     Turn  StartDistance  EndDistance  BrakingDuration  BrakingDistance  \
0      1     688.554468   791.561588         1.547558

## Compute Throttle features

In [6]:
try:
    throttle_segments_1 = analyze_throttle_segments(lap_1["Telemetry"], turns)
    throttle_segments_2 = analyze_throttle_segments(lap_2["Telemetry"], turns)
except Exception as e:
    print(f"Throttle segments analysis failed: {e}")


print(f"Throttle segments for {driver_1}:\n\n: {throttle_segments_1}\n")
print(f"Throttle segments for {driver_2}:\n\n: {throttle_segments_2}\n")

Throttle segments for ALO:

:     Turn  StartDistance  EndDistance  MeanThrottle  StdThrottle  \
0      0       1.840340   718.865490     92.439242    22.131016   
1      1     718.865490   830.361033      5.627521     9.083618   
2      2     830.361033  1063.112431     89.605124    21.715153   
3      3    1063.112431  1197.159026     68.750381    18.856849   
4      4    1197.159026  1364.076590     72.431012    24.259375   
5      5    1364.076590  1521.509446     51.646625    27.207720   
6      6    1521.509446  1841.732500     86.283184    28.028742   
7      7    1841.732500  2265.842642     92.927210    18.472717   
8      8    2265.842642  2412.668486     50.843783    40.508237   
9      9    2412.668486  2743.644082     91.563908    22.461258   
10    10    2743.644082  2889.668982     25.674870    42.138655   
11    11    2889.668982  3146.600553     89.215506    19.926732   
12    12    3146.600553  3784.834879     84.586763    33.370435   
13    13    3784.834879  3947.33

## Compute gear features

In [7]:
try:
    gear_shifts_1 = analyze_gear_shifts(lap_1["Telemetry"], turns)
    gear_shifts_2 = analyze_gear_shifts(lap_2["Telemetry"], turns)
except Exception as e:
    print(f"Gear shifts analysis failed: {e}")

print(f"Gear shifts for {driver_1}:\n\n: {gear_shifts_1}\n")
print(f"Gear shifts for {driver_2}:\n\n: {gear_shifts_2}\n")

Gear shifts for ALO:

:     Turn  NumberOfUpshifts  NumberOfDownshifts  MinimumRPM  MaximumRPM  \
0      0                 1                   0       10363       11848   
1      1                 0                   3        9788       10691   
2      2                 1                   0        9537       11851   
3      3                 0                   0       10901       11998   
4      4                 0                   0       10766       11357   
5      5                 0                   1       10579       11447   
6      6                 2                   0       10360       11965   
7      7                 0                   0       10591       11776   
8      8                 0                   1        9786       10730   
9      9                 4                   1        9764       11636   
10    10                 0                   3        6129       10876   
11    11                 4                   0        6260       11780   
12    12      

In [8]:
lap_1["SpeedMetrics"] = speed_metrics_1
lap_1["BrakingZones"] = braking_zones_1 
lap_1["ThrottleSegments"] = throttle_segments_1
lap_1["GearShifts"] = gear_shifts_1


lap_2["SpeedMetrics"] = speed_metrics_2
lap_2["BrakingZones"] = braking_zones_2
lap_2["ThrottleSegments"] = throttle_segments_2
lap_2["GearShifts"] = gear_shifts_2

try:
    save_cache(year, grand_prix, segment, lap_1, lap_2)
except Exception as e:
    print(f"Failed to save lap cache: {e}")

Saving payload to C:\Users\joseg\OneDrive\Universidade\Outros\Projetos\f1-teammate-analysis\data\2025_japanese_grand_prix_Q1_ALO.pkl
Saved: C:\Users\joseg\OneDrive\Universidade\Outros\Projetos\f1-teammate-analysis\data\2025_japanese_grand_prix_Q1_ALO.pkl
Saving payload to C:\Users\joseg\OneDrive\Universidade\Outros\Projetos\f1-teammate-analysis\data\2025_japanese_grand_prix_Q1_STR.pkl
Saved: C:\Users\joseg\OneDrive\Universidade\Outros\Projetos\f1-teammate-analysis\data\2025_japanese_grand_prix_Q1_STR.pkl

Processed data for 2025 Japanese Grand Prix saved to cache.
Driver 1: ALO, Driver 2: STR, Segment: Q1
